# Video analytics with Intel® Deep Learning Streamer (DL Streamer): Car detection and color classification

Step-by-step **DL Streamer** pipelines on a traffic clip: **car detection** and **color classification**; **OpenVINO™** runs the IR models.
Short cells below — pipeline text lives in `utils.py`. Close the preview window when a step ends (expected). `run_visual()` keeps console output quiet; FPS still appears on the video.


## Pipeline stages

```mermaid
flowchart LR
  VF[Video feed] --> Decode[Decode]
  Decode --> Pre[Pre-Process]
  Pre --> Inf[Inference]
  Inf --> Post[Post-Process]
  Post --> Enc[Encode]
  Enc --> AV[Annotated video]
```


## Configuration

- **Precision** → IR under `car-model/{FP16|FP32|INT8}/deployment/.../model.xml`.
- **Devices** → `gvadetect` / `gvaclassify` `device=` (dropdown uses OpenVINO when available).
- **Remote paths** (optional): before **Apply**, set `export DLSTREAMER_WORKDIR=/your/.../2` and `export DLSTREAMER_MODEL_ROOT=/your/.../car-model` (or edit defaults in `utils.py`).
- **Decoder**: default is `decodebin3` (same as *Video Analytics with DL-StreamerV1*). If your GStreamer has no `decodebin3`, run `export GST_DECODEBIN=decodebin` before Jupyter.

Run the cell below (opens dropdowns); **Apply** refreshes env. If `import utils` fails, set the kernel working directory to this folder.


In [ ]:
from utils import show_config_widgets

show_config_widgets()


### Verify `gvadetect`


In [ ]:
from utils import check_gvadetect
check_gvadetect()


## DEBUG — remove this section later

Temporary checks: paths, files, GStreamer / DL Streamer binaries, and a sample pipeline string. Delete this heading and the next cell when you no longer need it.

In [ ]:
# DEBUG — remove this cell later
import os
import shutil
import subprocess
import sys
from pathlib import Path

# Ensure utils is importable (same folder as notebook)
for d in (Path.cwd(), Path.cwd() / "2"):
    if (d / "utils.py").is_file():
        if str(d) not in sys.path:
            sys.path.insert(0, str(d))
        break

import utils as u

u.apply_environment()

def _run(cmd: list[str], timeout: float = 15) -> tuple[int, str]:
    try:
        r = subprocess.run(
            cmd,
            capture_output=True,
            text=True,
            timeout=timeout,
        )
        out = (r.stdout or "") + (r.stderr or "")
        return r.returncode, out.strip()[:4000]
    except Exception as e:
        return -1, str(e)

print("=== Python ===", sys.executable, sys.version.split()[0])
print("=== CWD ===", Path.cwd())
print("=== Display (autovideosink / XImageSink needs this on X11) ===")
print("  DISPLAY=", os.environ.get("DISPLAY", "(unset)"))
print("  WAYLAND_DISPLAY=", os.environ.get("WAYLAND_DISPLAY", "(unset)"))
print("  XDG_SESSION_TYPE=", os.environ.get("XDG_SESSION_TYPE", "(unset)"))

for name in ("gst-launch-1.0", "gst-inspect-1.0"):
    p = shutil.which(name)
    print(f"=== which {name} ===", p or "(not found)")

if shutil.which("gst-launch-1.0"):
    rc, out = _run(["gst-launch-1.0", "--version"])
    print("=== gst-launch-1.0 --version ===", f"rc={rc}\n", out[:800])

for el in ("decodebin3", "decodebin", "gvadetect", "gvaclassify", "gvatrack"):
    rc, out = _run(["gst-inspect-1.0", el])
    ok = rc == 0
    print(f"=== gst-inspect {el} ===", "OK" if ok else f"FAIL rc={rc}")
    if not ok and out:
        print(out[:500])

print("=== Env (DL Streamer / overrides) ===")
for k in sorted(os.environ):
    if k.startswith(("DLSTREAMER_", "GST_", "OPENVINO", "OV_")):
        print(f"  {k}={os.environ[k]!r}")

print("=== Key paths (from utils / env) ===")
keys = (
    "WORKDIR",
    "VIDEO_DIR",
    "VIDEO_SRC",
    "MODEL_DIR",
    "DETECTION_MODEL",
    "CLASSIFICATION_MODEL",
    "DETECTION_DEVICE",
    "CLASSIFICATION_DEVICE",
    "PRECISION",
    "GST_DECODEBIN",
)
for k in keys:
    v = os.environ.get(k, "(unset)")
    exists = ""
    if k.endswith("MODEL") or k.endswith("SRC") or k in ("VIDEO_DIR",):
        p = Path(v) if v != "(unset)" else None
        if p is not None:
            exists = f"  [{'FILE' if p.is_file() else 'DIR' if p.is_dir() else 'MISSING'}]"
    print(f"  {k}={v}{exists}")

print("=== Sample pipeline (first step, needs DISPLAY for window) ===")
print(u.pipeline_raw_video())
print("=== Headless smoke (no window — decodes only) ===")
print(u.pipeline_raw_video_fakesink())


### 1. Decode + display

The next cell prints the exact string from **`utils.pipeline_raw_video()`** (resolved paths), then runs it.

```bash
gst-launch-1.0 \
  filesrc location=$VIDEO_SRC ! \
  decodebin3 ! \
  videoconvert ! \
  autovideosink sync=true
```

(`decodebin3` → `decodebin` if you set `GST_DECODEBIN`.)

**`GstXImageSink` / `gst_x_image_sink_handle_xevents` / “Output window”:** usually **no usable display** for `autovideosink` (SSH without X11, wrong `DISPLAY`, or headless). Paths/plugins can still be fine. Try **`ssh -Y`** to the host, or `export DISPLAY=:0` on the machine’s console, or run the **headless** cell below (`fakesink`) to confirm decode only.


In [ ]:
from utils import preview_and_run, pipeline_raw_video

preview_and_run(pipeline_raw_video)


### 1b. (Optional) Headless decode — no window

Same as §1 but **`utils.pipeline_raw_video_fakesink()`** — no display, decode only:

```bash
gst-launch-1.0 \
  filesrc location=$VIDEO_SRC ! \
  decodebin3 ! \
  videoconvert ! \
  fakesink sync=false
```

If section 1 fails with **XImageSink** but you only need to confirm **decode** works (no GUI), run this. It plays the file to **`fakesink`** until EOS (can take a few seconds).

In [ ]:
from utils import preview_and_run, pipeline_raw_video_fakesink

preview_and_run(pipeline_raw_video_fakesink, benchmark=True)


### 2. + FPS overlay (`gvafpscounter` → `autovideosink`)

**`utils.pipeline_fps()`** — adds `gvafpscounter` before the sink:

```bash
gst-launch-1.0 \
  filesrc location=$VIDEO_SRC ! \
  decodebin3 ! \
  videoconvert ! \
  gvafpscounter ! \
  autovideosink sync=true
```


In [ ]:
from utils import preview_and_run, pipeline_fps

preview_and_run(pipeline_fps)


### 3. Vehicle detection

**`utils.pipeline_detection()`** — inference step is **`gvadetect`** (car IR), then overlays:

```bash
gst-launch-1.0 \
  filesrc location=$VIDEO_SRC ! \
  decodebin3 ! \
  gvadetect model=$DETECTION_MODEL device=$DETECTION_DEVICE pre-process-backend=opencv ! \
  gvawatermark ! \
  gvafpscounter ! \
  videoconvert ! \
  autovideosink sync=true
```


In [ ]:
from utils import preview_and_run, pipeline_detection

preview_and_run(pipeline_detection)


### 4. Detection + color classification

**`utils.pipeline_detect_classify()`** — **`gvatrack`** + **`gvaclassify`** (color IR) after detection:

```bash
gst-launch-1.0 \
  filesrc location=$VIDEO_SRC ! \
  decodebin3 ! \
  gvadetect model=$DETECTION_MODEL device=$DETECTION_DEVICE pre-process-backend=opencv ! \
  gvatrack ! \
  gvaclassify model=$CLASSIFICATION_MODEL device=$CLASSIFICATION_DEVICE pre-process-backend=opencv reclassify-interval=$RECLASSIFY_INTERVAL ! \
  queue ! gvawatermark ! gvafpscounter ! \
  videoconvert ! \
  autovideosink sync=true
```


In [ ]:
from utils import preview_and_run, pipeline_detect_classify

preview_and_run(pipeline_detect_classify)


### 5. Benchmark: 2 streams (`fakesink`)

**`utils.pipeline_benchmark_2()`** — two **parallel** copies of the branch below in **one** `gst-launch-1.0` (space-separated).

One branch (same as `utils._branch`):

```bash
gst-launch-1.0 \
  filesrc location=$VIDEO_SRC ! decodebin3 ! \
  gvadetect model=$DETECTION_MODEL device=$DETECTION_DEVICE pre-process-backend=opencv ! \
  gvatrack ! \
  gvaclassify model=$CLASSIFICATION_MODEL device=$CLASSIFICATION_DEVICE pre-process-backend=opencv reclassify-interval=$RECLASSIFY_INTERVAL ! \
  queue ! gvafpscounter ! fakesink sync=false
```

The real command concatenates **two** such branches (no `\\` between them — just a space).


In [ ]:
from utils import preview_and_run, pipeline_benchmark_2

preview_and_run(pipeline_benchmark_2, benchmark=True)


### 6. Benchmark: 4 streams

**`utils.pipeline_benchmark_4()`** — identical branch as §5, repeated **four** times in one `gst-launch-1.0` (space-separated). One branch shape:

```bash
gst-launch-1.0 \
  filesrc location=$VIDEO_SRC ! decodebin3 ! \
  gvadetect model=$DETECTION_MODEL device=$DETECTION_DEVICE pre-process-backend=opencv ! \
  gvatrack ! \
  gvaclassify model=$CLASSIFICATION_MODEL device=$CLASSIFICATION_DEVICE pre-process-backend=opencv reclassify-interval=$RECLASSIFY_INTERVAL ! \
  queue ! gvafpscounter ! fakesink sync=false
```

The runnable pipeline string (four branches) is printed when you run the next cell.


In [ ]:
from utils import preview_and_run, pipeline_benchmark_4

preview_and_run(pipeline_benchmark_4, benchmark=True)
